<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2002%20-%20Numbers%20Become%20Vectors/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 02 — Numbers Become Vectors · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Four house sales, now with floor area recorded:

| House | Rooms | Area (sq ft) | Price (₹ lakh) |
|---|---:|---:|---:|
| A | 2 | 800 | 9 |
| B | 2 | 1200 | 11 |
| C | 3 | 900 | 11.5 |
| D | 4 | 1600 | 17 |

**A and B have the same number of rooms and differ by ₹2 lakh.** Chapter 1's model takes
one input and returns one output, so it must give them the same price. Step 3 makes it try.

## Step 2 — Prediction

Commit to answers before running anything. You check them in Step 10.

1. Train Chapter 1's one-input model (rooms only) on these four houses. What is the smallest
   loss it can possibly reach — zero, or something stubbornly above zero?
2. Houses $\mathbf{p}=[1,4]$ and $\mathbf{q}=[2,8]$ are the same kind of flat at double the
   size. Is their cosine similarity closer to $0$, $0.5$, or $1$? Is their *distance* small?
3. Rooms live in 2–4; area lives in 800–1600. Will gradient descent on the raw features be
   slow, explosive, or fine?
4. Is there **any** learning rate that trains the raw features well in 2000 steps?

In [ ]:
# Step 3 — Intuition: watch Chapter 1's model fail, for a reason training cannot fix.
import numpy as np
np.random.seed(0)

rooms  = np.array([2., 2., 3., 4.])
area   = np.array([800., 1200., 900., 1600.])
prices = np.array([9., 11., 11.5, 17.])

def train_1d(x, y, learning_rate=0.05, steps=5000):
    w = b = 0.0
    for _ in range(steps):
        error = (w * x + b) - y
        w -= learning_rate * np.mean(2 * error * x)
        b -= learning_rate * np.mean(2 * error)
    return w, b

# Rooms are 2-4, so this is numerically safe at this learning rate.
w1, b1 = train_1d(rooms / 4, prices)      # scaled to keep it well behaved
pred = w1 * (rooms / 4) + b1
print("predictions:", np.round(pred, 3))
print("actual:     ", prices)
print(f"best possible loss with rooms alone = {np.mean((pred - prices) ** 2):.4f}")

# A and B have identical inputs, so the model MUST give them identical prices.
assert abs(pred[0] - pred[1]) < 1e-9
# ...and therefore can never reach zero loss on data where they differ.
assert np.mean((pred - prices) ** 2) > 1.0
print("\nA and B get the same price. No value of w and b can change that.")

## Step 4 — The Mathematics Under Test

One measurement per house becomes $d$ measurements, held in one object:

$$\hat{y} = \mathbf{w}\cdot\mathbf{x} + b
\qquad
\mathbf{w}\cdot\mathbf{x} = \sum_{i=1}^{d} w_i x_i
\qquad
\|\mathbf{x}\| = \sqrt{\mathbf{x}\cdot\mathbf{x}}
\qquad
\cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$$

Step 5 checks every number the lecture claims.

In [ ]:
# Step 5 — Manual calculation: the dot product prices every house (blog section 7).
X = np.column_stack([rooms, area])      # one row per house, one column per feature
w = np.array([2.0, 0.005])              # ₹2 lakh per room, ₹0.005 lakh per sq ft
b = 1.0
print("X shape:", X.shape, " w shape:", w.shape)

# House A by hand, exactly as section 7 writes it:
a = X[0]
print(f"\nw . x_A = {w[0]}*{a[0]} + {w[1]}*{a[1]} = {w[0]*a[0]} + {w[1]*a[1]} = {w @ a}")
print(f"price_A = {w @ a} + {b} = {w @ a + b}")
assert w @ a == 8.0
assert w @ a + b == 9.0

# All four at once — the same formula, unchanged:
predictions = X @ w + b
print("\npredictions:", predictions)
assert np.allclose(predictions, prices)
print("every house priced exactly right, with one rule")

# Two vectors in, ONE number out. That is why the dot product is the right tool.
assert np.isscalar(w @ a) or (w @ a).ndim == 0

In [ ]:
# Step 5b — the distinction that bites everyone once (blog section 13).
u = np.array([2.0, 800.0])
v = np.array([1.0, 400.0])

print("u * v  (element-wise) =", u * v, "  -> shape", (u * v).shape)
print("u @ v  (dot product)  =", u @ v, "         -> a single number")
assert (u * v).shape == (2,)
assert np.ndim(u @ v) == 0
assert u @ v == (u * v).sum()      # the dot product is the sum of the element-wise product

# Addition and scaling (section 6)
assert np.array_equal(np.array([2., 800.]) + np.array([1., 400.]), np.array([3., 1200.]))
assert np.array_equal(2 * np.array([2., 800.]), np.array([4., 1600.]))

In [ ]:
# Step 5c — length, distance, and the angle result derived in section 9.
def norm(v):      return np.sqrt(v @ v)          # section 8: sqrt of the dot with itself
def cosine(a, c): return (a @ c) / (norm(a) * norm(c))

assert norm(np.array([3.0, 4.0])) == 5.0         # Pythagoras
assert np.isclose(norm(np.array([3., 4.])), np.linalg.norm(np.array([3., 4.])))

p = np.array([1.0, 4.0])    # 1 room,  400 sq ft
q = np.array([2.0, 8.0])    # 2 rooms, 800 sq ft  — same shape, double the size
r = np.array([4.0, 1.0])    # 4 rooms, 100 sq ft  — a different kind of building

print(f"distance(p, q) = {norm(p - q):.4f}   (sqrt 17)")
print(f"distance(p, r) = {norm(p - r):.4f}   (sqrt 18)")
print(f"cosine(p, q)   = {cosine(p, q):.4f}")
print(f"cosine(p, r)   = {cosine(p, r):.4f}   (8/17)")

assert np.isclose(norm(p - q), np.sqrt(17))
assert np.isclose(norm(p - r), np.sqrt(18))
assert np.isclose(cosine(p, q), 1.0)            # identical direction
assert np.isclose(cosine(p, r), 8 / 17)

# Distance says p is about equally close to both. Cosine says p and q are the
# same kind of house and r is not. They answer different questions.

# The boxed result of section 9: u . v == |u| |v| cos(theta)
theta = np.arccos(np.clip(cosine(p, r), -1, 1))
assert np.isclose(p @ r, norm(p) * norm(r) * np.cos(theta))
print(f"\nangle between p and r = {np.degrees(theta):.2f} degrees")

# Orthogonal vectors: a dot product of exactly zero
assert np.array([1.0, 0.0]) @ np.array([0.0, 1.0]) == 0.0

In [ ]:
# Step 6 — First implementation: gradient descent with vectors instead of scalars.
# Identical to Chapter 1's loop; only the shapes changed.

def train(X, y, learning_rate=0.1, steps=2000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(steps):
        error = (X @ w + b) - y                  # 1-2. predict and measure
        w -= learning_rate * (2 / n) * (X.T @ error)   # 3-4. slopes, then step
        b -= learning_rate * 2 * error.mean()
        if not np.all(np.isfinite(w)):
            return w, b, float("inf")
    return w, b, np.mean(((X @ w + b) - y) ** 2)

# The gradient line above is the vector form of Chapter 1's  dw = mean(2*error*x).
# Written out one feature at a time, it is exactly the same sum:
def gradient_explicit(X, y, w, b):
    n, d = X.shape
    g = np.zeros(d)
    for j in range(d):                  # for each feature
        for i in range(n):              # for each house
            g[j] += 2 * ((X[i] @ w + b) - y[i]) * X[i, j]
    return g / n

w_test, b_test = np.array([1.0, 0.001]), 0.5
err = (X @ w_test + b_test) - prices
assert np.allclose(gradient_explicit(X, prices, w_test, b_test), (2 / len(X)) * (X.T @ err))
print("explicit loop and vectorized gradient agree")

In [ ]:
# Step 7 — Visualization: houses as points, and what cosine sees that distance does not.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) the four houses as points on a map (blog section 5)
axes[0].scatter(rooms, area, s=80, zorder=3)
for name, rm, ar in zip("ABCD", rooms, area):
    axes[0].annotate(f" {name}", (rm, ar), fontsize=12)
axes[0].plot([2, 2], [800, 1200], "--", color="crimson", lw=1)
axes[0].set_xlabel("rooms"); axes[0].set_ylabel("area (sq ft)")
axes[0].set_title("A and B: same rooms, 400 sq ft apart")

# (b) p, q, r as arrows — direction is the thing that differs
for vec, name, colour in [(p, "p", "tab:blue"), (q, "q", "tab:green"), (r, "r", "crimson")]:
    axes[1].arrow(0, 0, vec[0], vec[1], head_width=.2, length_includes_head=True, color=colour)
    axes[1].annotate(name, vec * 1.05, color=colour, fontsize=13)
axes[1].set_xlim(-.5, 5); axes[1].set_ylim(-.5, 9)
axes[1].set_title("p and q share a direction; r does not")
axes[1].set_xlabel("rooms"); axes[1].set_ylabel("area (100 sq ft)")

# (c) the loss landscape is a canyon when features are unscaled (section 10)
w_r = np.linspace(-2, 6, 120)
w_a = np.linspace(-0.02, 0.03, 120)
WR, WA = np.meshgrid(w_r, w_a)
Z = np.zeros_like(WR)
for i in range(WR.shape[0]):
    for j in range(WR.shape[1]):
        Z[i, j] = np.mean((rooms * WR[i, j] + area * WA[i, j] + 1 - prices) ** 2)
axes[2].contour(WR, WA, np.log10(Z + 1e-9), levels=25)
axes[2].plot(2, 0.005, "*", ms=18, color="crimson")
axes[2].set_xlabel("weight on rooms"); axes[2].set_ylabel("weight on area")
axes[2].set_title("log-loss: a canyon, not a bowl")

for ax in axes:
    ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The controlled experiment: measure the shape of the landscape.
# Chapter 1 section 10 showed steepness along a feature is governed by mean(x^2).

print(f"mean(rooms^2) = {np.mean(rooms ** 2):>12,.2f}")
print(f"mean(area^2)  = {np.mean(area ** 2):>12,.2f}")
print(f"ratio         = {np.mean(area ** 2) / np.mean(rooms ** 2):>12,.0f}x")
assert np.mean(rooms ** 2) == 8.25
assert np.mean(area ** 2) == 1362500.0

def condition_number(M):
    """Ratio of steepest to flattest direction of the loss surface."""
    H = 2 / len(M) * (M.T @ M)          # curvature of MSE in the weights
    eigenvalues = np.linalg.eigvalsh(H)
    return eigenvalues[-1] / eigenvalues[0], eigenvalues[-1], eigenvalues[0]

X_scaled = X / X.max(axis=0)            # divide each feature by its largest value
cond_raw,    hi_raw,    lo_raw    = condition_number(X)
cond_scaled, hi_scaled, lo_scaled = condition_number(X_scaled)

print(f"\nraw:    steepest {hi_raw:>14,.2f}   flattest {lo_raw:>8.4f}   condition {cond_raw:>12,.0f}")
print(f"scaled: steepest {hi_scaled:>14,.2f}   flattest {lo_scaled:>8.4f}   condition {cond_scaled:>12,.1f}")
assert cond_raw > 1e6 and cond_scaled < 200

# Predict the stability limit BEFORE training, as Chapter 1 section 14 did.
# The bias is a third direction, so include a column of ones.
def stability_limit(M):
    A = np.column_stack([M, np.ones(len(M))])
    H = 2 / len(A) * (A.T @ A)
    return 2 / np.linalg.eigvalsh(H)[-1]

print(f"\npredicted largest safe eta — raw:    {stability_limit(X):.2e}")
print(f"predicted largest safe eta — scaled: {stability_limit(X_scaled):.4f}")
assert abs(stability_limit(X_scaled) - 0.4997) < 1e-3

In [ ]:
# Step 9 — Change exactly one thing: the scale of the features. Same data, same loop.
print("RAW FEATURES")
for eta in [1e-3, 1e-5, 1e-6, 7e-7]:
    with np.errstate(over="ignore", invalid="ignore"):
        w_r_, b_r_, loss_r_ = train(X, prices, learning_rate=eta, steps=2000)
    state = "diverged" if not np.isfinite(loss_r_) else f"loss={loss_r_:.4f}  w={np.round(w_r_, 4)}"
    print(f"  eta={eta:<8} {state}")

print("\nSCALED FEATURES (each divided by its maximum)")
for eta in [0.1, 0.45, 0.5]:   # the predicted limit sits at 0.4997
    w_s_, b_s_, loss_s_ = train(X_scaled, prices, learning_rate=eta, steps=2000)
    print(f"  eta={eta:<8} loss={loss_s_:.9f}  w={np.round(w_s_, 4)}  b={b_s_:.4f}")

# Recover the real-world weights by undoing the scaling.
w_s, b_s, loss_s = train(X_scaled, prices, learning_rate=0.1, steps=2000)
w_recovered = w_s / X.max(axis=0)
print(f"\nrecovered w = {w_recovered}   (true: [2.0, 0.005])")
print(f"recovered b = {b_s:.4f}                (true: 1.0)")
assert loss_s < 1e-5
assert np.allclose(w_recovered, [2.0, 0.005], atol=1e-3)
assert abs(b_s - 1.0) < 1e-2

## Step 10 — Observe

Against your Step 2 predictions:

- Chapter 1's model bottoms out at a loss above **1.2** and cannot go lower. A and B receive
  identical prices because they have identical inputs. This is a limit of the
  *representation*, not of the training.
- $\cos(\mathbf{p}, \mathbf{q}) = 1$ **exactly**, while their distance is $\sqrt{17} \approx 4.12$ —
  farther apart than you might guess. Same character, different size.
- The raw features have a condition number over **3.6 million**. Every learning rate large
  enough to move the rooms weight explodes along the area direction; $\eta = 7\times10^{-7}$
  survives but leaves the weights near zero after 2000 steps.
- So the answer to question 4 is **no**. There is no good learning rate for the raw features.
  The problem is not the optimizer.
- Scaling both features drops the condition number to about **85**. The predicted stability
  limit is $2/4.0028 = 0.4997$, and the runs bracket it exactly: $ta = 0.45$ converges,
  $ta = 0.5$ blows up. With $ta = 0.1$ training
  recovers $[2.0,\, 0.005]$ and $b = 1$ — the true rule, in real units, after un-scaling.

## Step 11 — Explain

Chapter 1 established that the steepness of the loss valley along a feature is set by
$\frac{1}{n}\sum x_i^2$. Here those numbers are $8.25$ for rooms and $1{,}362{,}500$ for area —
a factor of about 165,000.

The learning rate is a **single number applied to every direction at once**. It must be
small enough to survive the steepest direction, or the update overshoots and compounds,
exactly as in Chapter 1 §14. But that same step, taken along the nearly flat rooms
direction, moves that weight by almost nothing. One number cannot serve both.

So the loss surface is not a bowl — it is a canyon, which is what panel (c) of Step 7 shows.
Gradient descent bounces across the narrow direction while crawling along the long one.

Dividing each feature by its maximum does not change what the data *means*: the same rule
is recoverable, as the un-scaling in Step 9 proves. It changes the **geometry the optimizer
walks on** — and Chapter 1 taught us that the geometry is the whole problem.

> This is the honest reason every serious model normalizes its inputs. Not convention.
> Conditioning.

In [ ]:
# Step 12 — Challenges.

# LEVEL 4 (Investigate), part 1:
# Find the largest learning rate that still converges for raw features, and for scaled.
# Then check: does the ratio of those two thresholds match the ratio of the steepest
# curvature (hi_raw / hi_scaled) computed in Step 8?

# YOUR CODE HERE


# LEVEL 4, part 2:
# Scale ONLY the area, leaving rooms raw. Does partial scaling help, hurt, or do nothing?
# Compute the condition number first and predict the answer before training.
X_partial = X.copy()
X_partial[:, 1] = X_partial[:, 1] / X_partial[:, 1].max()

# YOUR CODE HERE


# LEVEL 5 (Design):
# The office adds a neighbourhood: 'Andheri', 'Bandra', 'Colaba'.
# Encoding them 1, 2, 3 tells the geometry that Bandra is midway between the other two
# and that Andheri + Colaba = 2 x Bandra. Both are nonsense.
# Design an encoding that does not lie. How many slots does it need? What is the distance
# between any two neighbourhoods under your scheme? What breaks at 50,000 neighbourhoods?

neighbourhoods = ["Andheri", "Bandra", "Colaba"]

def encode(name):
    # YOUR CODE HERE
    ...

## Step 13 — Reflection

- [ ] I can state why Chapter 1's model fails on houses A and B without using the word "training".
- [ ] I can explain why `u * v` and `u @ v` return different *shapes*, and which one prices a house.
- [ ] I derived $\mathbf{u}\cdot\mathbf{v} = \|\mathbf{u}\|\|\mathbf{v}\|\cos\theta$ by hand and watched Step 5c confirm it.
- [ ] I can say when I want distance and when I want cosine, with an example of each.
- [ ] I can explain the canyon in Step 7(c) using the two numbers printed in Step 8.
- [ ] I understand that scaling changed the geometry, not the data — and can prove it by un-scaling.

### The question this chapter leaves open

One house is one vector, and one dot product prices it. Four houses take four dot products;
a million take a million. And in Chapter 5 a network will want **many different weight
vectors at once** — one per neuron, each asking a different question about the same house.

Stack vectors into rows and you get a table of numbers with its own algebra.

➡️ **Next:** [Chapter 03 — Matrices: The Spreadsheet of Mathematics](<../Lecture 03 - Matrices: The Spreadsheet of Mathematics/blog.md>)